# 4D 랑 3D 모두 gaussian_dim== 이랑 관련 X  
- 4D to 3D convert 하는법  
    - if(s_t < tau)

In [ ]:
s_t = gaussianModel.scaling_t
tau = 0.05
if(s_t < tau):
    gaussian.convert_to_static()
    # time related variable 모두 다 discard or detach 
    # static_attributes 에 저장
else:
    pass # retain 4D

In [ ]:
# rot_4D boolean 이 있는데
# 이 값에 따라서 forward.cu 내부에서 computeCov4D 할지 아니면 3D 할 지를 결정함
# 그럼 이걸 갖고와서 static 과 dynamic 을 나눌 수 있을듯..?

# static_attribute 로 옮기는 거 

marginal_t 는 t 에 따라 달라지는 1D 분포(가우시안) → mean_t 랑 scale_t 를 이용해서 분포 만들고 거기서 t 로 sampling

In [ ]:
def get_marginal_t(self, timestamp, scaling_modifier = 1): # Standard
        sigma = self.get_cov_t(scaling_modifier)
        return torch.exp(-0.5*(self.get_t-timestamp)**2/sigma) # / torch.sqrt(2*torch.pi*sigma)

입력변수로 time(timestamp) 가 들어감

그거에따른 margianl_t 를 구함.(시간에따라달라짐)

scale_t 는 고정, marginal_t 는 scale_t 에 따라 달라짐

In [ ]:
    @property
    def get_xyz(self):
        return self._xyz
    
    @property
    def get_t(self):
        return self._t
    
    @property
    def get_xyzt(self):
        return torch.cat([self._xyz, self._t], dim = 1)

- create_from_pcd 함수 체크하기

1. static_attributes 들로 값 옮기기
2. 기존의 time related attributes 들 detach or discard
3. 나중에 합칠 때 어떻게 합칠 지??
    - 3D 4D 합친 다음에 걔네들을 연산할 때, rot_4d boolean 을 통해서 branch 나눠주면 될 거 같기는 함.
    

In [ ]:
def get_combined_means(self, timestamp):
    # 동적 Gaussian: 시간 offset(delta_mean)을 적용해 현재 시간에 맞는 3D 위치 계산
    dynamic_means = self._xyz.clone()
    if self.gaussian_dim == 4:
        _, delta_mean = self.get_current_covariance_and_mean_offset(1.0, timestamp)
        if hasattr(self, 'static_mask') and self.static_mask.numel() > 0:
            delta_mean[self.static_mask] = 0.0  # 정적 부분은 시간 오프셋 무시
        dynamic_means += delta_mean
    # 정적 Gaussian은 이미 3D로 변환되어 static_xyz에 저장됨
    static_means = self.static_xyz
    return torch.cat([dynamic_means, static_means], dim=0)

- **scene/gaussian_model.py**
    - 생성자에서 정적 파라미터(예: static_xyz, static_features_dc 등)를 초기화
    - `capture()`와 `restore()` 함수에 정적 파라미터 포함
    - `separate_static_gaussians`(tau) (또는 convert_to_3d(indices)) 함수를 구현하여, exp⁡(st)>τ\exp(s_t) > \tauexp(st)>τ인 4D Gaussian을 3D로 전환
    - `get_combined_means`(), `get_combined_covariance`() (및 필요 시 get_all_opacities(), get_all_sh_features()) 함수를 추가하여 동적과 정적 데이터를 하나로 결합
- **train.py**
    - 학습 루프 내에서 densification 또는 opacity reset 후 주기적으로 `separate_static_gaussians`(tau)를 호출하여 정적 Gaussian 전환을 수행
- **gaussian_renderer (예: init.py)**
    - 렌더링 시 모델의 통합 getter 함수들을 사용하여, 동적 및 정적 Gaussian의 데이터를 하나로 결합한 후 투영, 깊이 정렬, 그리고 alpha blending을 수행

- 4D 랑 3D 나눈 다음에, 4D -> 3D projection 시키고 나머지 static 3D 랑 합친 뒤에 2D 로 projection 시켜서 해보기
- 어디서 합칠지
- p(t) marginal time == scale_t 인가??


# 1. scene/gaussian_model.py

In [ ]:
class GaussianModel:
    def __init__(self, sh_degree: int, gaussian_dim: int = 3, time_duration: list = [-0.5, 0.5],
                 rot_4d: bool = False, force_sh_3d: bool = False, sh_degree_t: int = 0):
        # 기존 4D 파라미터 초기화 (예시)
        self.gaussian_dim = gaussian_dim
        self._xyz = torch.empty(0)
        self._features_dc = torch.empty(0)
        self._features_rest = torch.empty(0)
        self._scaling = torch.empty(0)
        self._rotation = torch.empty(0)
        self._opacity = torch.empty(0)
        self.max_radii2D = torch.empty(0)
        self.xyz_gradient_accum = torch.empty(0)
        self.denom = torch.empty(0)
        self.optimizer = None
        self.percent_dense = 0
        self.spatial_lr_scale = 0

        self._t = torch.empty(0)
        self._scaling_t = torch.empty(0)
        self.time_duration = time_duration
        self.rot_4d = rot_4d
        self._rotation_r = torch.empty(0)
        self.force_sh_3d = force_sh_3d
        self.t_gradient_accum = torch.empty(0)
        self.env_map = torch.empty(0)

        self.active_sh_degree = 0
        self.max_sh_degree = sh_degree
        self.active_sh_degree_t = 0
        self.max_sh_degree_t = sh_degree_t

        self.setup_functions()

        # ===== 추가: 정적 Gaussian 파라미터 초기화 =====
        self.static_xyz = torch.empty(0)
        self.static_features_dc = torch.empty(0)
        self.static_features_rest = torch.empty(0)
        self.static_scaling = torch.empty(0)
        self.static_rotation = torch.empty(0)
        self.static_opacity = torch.empty(0)
        self.static_max_radii2D = torch.empty(0)
        self.static_xyz_gradient_accum = torch.empty(0)
        # =================================================

def capture(self):
    if self.gaussian_dim == 4:
        return (
            self.active_sh_degree,
            self._xyz,
            self._features_dc,
            self._features_rest,
            self._scaling,
            self._rotation,
            self._opacity,
            self.max_radii2D,
            self.xyz_gradient_accum,
            self.t_gradient_accum,
            self.denom,
            self.optimizer.state_dict(),
            self.spatial_lr_scale,
            self._t,
            self._scaling_t,
            self._rotation_r,
            self.rot_4d,
            self.env_map,
            self.active_sh_degree_t,
            # ===== 추가: 정적 파라미터 =====
            self.static_xyz,
            self.static_features_dc,
            self.static_features_rest,
            self.static_scaling,
            self.static_rotation,
            self.static_opacity,
            self.static_max_radii2D,
            self.static_xyz_gradient_accum
            # =================================
        )
    else:
        # 3D인 경우는 기존 방식 사용
        pass

def restore(self, model_args, training_args):
    if self.gaussian_dim == 4:
        (self.active_sh_degree,
         self._xyz,
         self._features_dc,
         self._features_rest,
         self._scaling,
         self._rotation,
         self._opacity,
         self.max_radii2D,
         self.xyz_gradient_accum,
         self.t_gradient_accum,
         self.denom,
         opt_dict,
         self.spatial_lr_scale,
         self._t,
         self._scaling_t,
         self._rotation_r,
         self.rot_4d,
         self.env_map,
         self.active_sh_degree_t,
         # ===== 추가: 정적 파라미터 복원 =====
         self.static_xyz,
         self.static_features_dc,
         self.static_features_rest,
         self.static_scaling,
         self.static_rotation,
         self.static_opacity,
         self.static_max_radii2D,
         self.static_xyz_gradient_accum
         # ==================================
        ) = model_args
    else:
        # 3D인 경우는 기존 방식 사용
        pass

    if training_args is not None:
        self.training_setup(training_args)
        self.xyz_gradient_accum = self.xyz_gradient_accum
        self.t_gradient_accum = self.t_gradient_accum
        self.denom = self.denom
        self.optimizer.load_state_dict(opt_dict)


def separate_static_gaussians(self, tau):
    """
    4D Gaussian 중 _scaling_t의 exp() 값이 tau를 초과하면 해당 Gaussian을 정적으로 전환합니다.
    전환 시, 시간 성분(_t)을 0으로 설정하고, 4D 회전 행렬에서 공간 부분만 추출하며, 
    시간 관련 파라미터(_scaling_t, 시간 SH 등)는 고정합니다.
    """
    if self.gaussian_dim != 4:
        print("모델이 3D이므로 정적/동적 분리는 적용되지 않습니다.")
        return

    temporal_scales = torch.exp(self._scaling_t).squeeze(-1)  # (N,)
    static_mask = temporal_scales > tau
    dynamic_mask = ~static_mask

    num_static = static_mask.sum().item()
    num_dynamic = dynamic_mask.sum().item()
    print(f"정적 Gaussian 개수: {num_static}, 동적 Gaussian 개수: {num_dynamic}")
    if num_static == 0:
        return

    # 정적 Gaussian 파라미터를 추가
    self.static_xyz = torch.cat([self.static_xyz, self._xyz[static_mask]], dim=0)
    self.static_features_dc = torch.cat([self.static_features_dc, self._features_dc[static_mask]], dim=0)
    self.static_features_rest = torch.cat([self.static_features_rest, self._features_rest[static_mask]], dim=0)
    self.static_scaling = torch.cat([self.static_scaling, self._scaling[static_mask]], dim=0)
    self.static_rotation = torch.cat([self.static_rotation, self._rotation[static_mask]], dim=0)
    self.static_opacity = torch.cat([self.static_opacity, self._opacity[static_mask]], dim=0)
    self.static_max_radii2D = torch.cat([self.static_max_radii2D, self.max_radii2D[static_mask]], dim=0)
    self.static_xyz_gradient_accum = torch.cat([self.static_xyz_gradient_accum, self.xyz_gradient_accum[static_mask]], dim=0)

    # 동적 Gaussian 집합에서 정적 항목 제거 (시간 관련 파라미터도 함께 제거)
    self._xyz = self._xyz[dynamic_mask]
    self._features_dc = self._features_dc[dynamic_mask]
    self._features_rest = self._features_rest[dynamic_mask]
    self._scaling = self._scaling[dynamic_mask]
    self._rotation = self._rotation[dynamic_mask]
    self._opacity = self._opacity[dynamic_mask]
    self.max_radii2D = self.max_radii2D[dynamic_mask]
    self.xyz_gradient_accum = self.xyz_gradient_accum[dynamic_mask]
    self._t = self._t[dynamic_mask]
    self._scaling_t = self._scaling_t[dynamic_mask]
    if self.rot_4d:
        self._rotation_r = self._rotation_r[dynamic_mask]

def get_combined_means(self, timestamp):
    """
    주어진 timestamp에서 동적 Gaussian은 시간 offset을 적용하여 3D 좌표를 계산하고,
    정적 Gaussian은 이미 3D 상태이므로 그대로 반환한 후, 두 그룹을 합쳐 반환한다.
    """
    dynamic_means = self._xyz.clone()
    if self.gaussian_dim == 4:
        _, delta_mean = self.get_current_covariance_and_mean_offset(1.0, timestamp)
        if hasattr(self, 'static_mask') and self.static_mask.any():
            delta_mean[self.static_mask] = 0.0
        dynamic_means += delta_mean
    # 정적 Gaussian의 좌표는 self.static_xyz에 저장되어 있음
    static_means = self.static_xyz
    return torch.cat([dynamic_means, static_means], dim=0)

def get_combined_covariance(self, scaling_modifier=1.0, timestamp=None):
    """
    주어진 timestamp에서 동적 Gaussian과 정적 Gaussian의 3×3 공분산을 결합하여 반환한다.
    """
    if self.gaussian_dim == 4:
        cov, _ = self.get_current_covariance_and_mean_offset(scaling_modifier, timestamp)
        if hasattr(self, 'static_mask') and self.static_mask.any():
            static_idx = self.static_mask
            cov_static = GaussianModel.build_covariance_from_scaling_rotation(
                scaling_modifier * self._scaling[static_idx],
                torch.eye(3, device=cov.device).reshape(1, 3, 3).repeat(static_idx.sum(), 1, 1)
            )
            cov[static_idx] = cov_static
        dynamic_cov = cov
    else:
        dynamic_cov = self.get_covariance(scaling_modifier)
    
    # 정적 Gaussian의 공분산 계산 (이미 3D이므로)
    static_cov = GaussianModel.build_covariance_from_scaling_rotation(
        self.static_scaling.exp(),  # scaling_activation 적용된 값
        self.static_rotation
    )
    return torch.cat([dynamic_cov, static_cov], dim=0)


# # ???
# def get_all_opacities(self):
#     dynamic_opacity = self.opacity_activation(self._opacity)
#     static_opacity = self.opacity_activation(self.static_opacity)
#     return torch.cat([dynamic_opacity, static_opacity], dim=0)

# def get_all_sh_features(self):
#     dynamic_sh = torch.cat((self._features_dc, self._features_rest), dim=2)
#     static_sh = torch.cat((self.static_features_dc, self.static_features_rest), dim=2)
#     return torch.cat([dynamic_sh, static_sh], dim=0)



# 2. train.py

In [ ]:
# train.py (학습 루프 내, densification 단계 후)
if iteration > opt.initial_warmup and iteration % opt.static_conversion_interval == 0:
    # 예: opt.static_conversion_interval = 100, opt.initial_warmup = 500
    # tau는 예를 들어 3.0 (또는 데이터셋에 맞게 조정)
    model.separate_static_gaussians(tau)

# 학습 루프 내에서 densification이나 opacity reset 후, 주기적으로 정적/동적 분리 함수를 호출하여 4D Gaussian 중 exp(st) > tau 인 항목을 3D로 전환합니다.

# 3. gaussian_renderer/init.py

설명:

렌더링 함수에서 동적와 정적 Gaussian 데이터를 모두 결합한 후, 단일 GPU rasterizer에서 투영, 정렬, 그리고 alpha blending을 수행합니다.

동적 Gaussian와 정적 Gaussian의 3D 좌표, 공분산, 불투명도, SH 특징을 각각 통합(get_combined_means, get_combined_covariance, get_all_opacities, get_all_sh_features)하여 하나의 리스트로 합칩니다.

이를 GPU rasterizer에 전달하면, 동일한 투영 및 alpha blending 과정을 통해 하나의 최종 이미지가 생성됩니다.

In [ ]:
# __init__.py 또는 렌더링 함수 내
# 'pc'는 GaussianModel 인스턴스, 'viewpoint_camera.timestamp'는 현재 렌더링 시간

# 1. 모든 Gaussian의 3D 좌표 결합
means_all = pc.get_combined_means(viewpoint_camera.timestamp)  # [N_dynamic + N_static, 3]

# 2. 모든 Gaussian의 공분산 결합
cov_all = None
if pipe.compute_cov3D_python:
    cov_all = pc.get_combined_covariance(scaling_modifier, viewpoint_camera.timestamp)

# 3. 불투명도 및 SH 특징도 통합 (필요 시)
opacity_all = pc.get_all_opacities()  # 동적 + 정적
sh_all = eval_shfs_4d(pc.active_sh_degree, pc.active_sh_degree_t, pc.get_features(), 
                      viewpoint_camera.get_ray_directions(), t=viewpoint_camera.timestamp)
# 4. rasterization: 기존 rasterizer에 통합 데이터를 넘겨서 렌더링
raster_settings = GaussianRasterizationSettings(
    image_height = viewpoint_camera.image_height,
    image_width = viewpoint_camera.image_width,
    tanfovx = math.tan(viewpoint_camera.FoVx * 0.5),
    tanfovy = math.tan(viewpoint_camera.FoVy * 0.5),
    bg = bg_color_tensor,
    scale_modifier = scaling_modifier,
    viewmatrix = viewpoint_camera.world_view_transform,
    projmatrix = viewpoint_camera.full_proj_transform,
    sh_degree = pc.active_sh_degree,
    sh_degree_t = pc.active_sh_degree_t,
    campos = viewpoint_camera.camera_center,
    timestamp = viewpoint_camera.timestamp,
    time_duration = pc.time_duration[1] - pc.time_duration[0],
    rot_4d = pc.rot_4d,
    gaussian_dim = pc.gaussian_dim,
    force_sh_3d = pc.force_sh_3d,
    prefiltered = False,
    opa_threshold = pipe.opa_threshold,
    debug = pipe.debug
)
rasterizer = GaussianRasterizer(raster_settings)
image = rasterizer(means3D = means_all, cov3D_precomp = cov_all, colors = sh_all, opacity = opacity_all)



# _ _ init _ _ 

In [ ]:
#
# Copyright (C) 2023, Inria
# GRAPHDECO research group, https://team.inria.fr/graphdeco
# All rights reserved.
#
# This software is free for non-commercial, research and evaluation use 
# under the terms of the LICENSE.md file.
#
# For inquiries contact  george.drettakis@inria.fr
#

import torch
from torch.nn import functional as F
import math
from .diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer
from scene.gaussian_model import GaussianModel
from utils.sh_utils import eval_sh, eval_shfs_4d
from collections import defaultdict

def render(viewpoint_camera, pc : GaussianModel, pipe, bg_color : torch.Tensor, scaling_modifier = 1.0, override_color = None):
    """
    Render the scene. 
    
    Background tensor (bg_color) must be on GPU!
    """
 
    # Create zero tensor. We will use it to make pytorch return gradients of the 2D (screen-space) means
    screenspace_points = torch.zeros_like(pc.get_xyz, dtype=pc.get_xyz.dtype, requires_grad=True, device="cuda") + 0
    try:
        screenspace_points.retain_grad()
    except:
        pass

    # Set up rasterization configuration
    tanfovx = math.tan(viewpoint_camera.FoVx * 0.5)
    tanfovy = math.tan(viewpoint_camera.FoVy * 0.5)

    raster_settings = GaussianRasterizationSettings(
        image_height=int(viewpoint_camera.image_height),
        image_width=int(viewpoint_camera.image_width),
        tanfovx=tanfovx,
        tanfovy=tanfovy,
        bg=bg_color if not pipe.env_map_res else torch.zeros(3, device="cuda"),
        scale_modifier=scaling_modifier,
        viewmatrix=viewpoint_camera.world_view_transform,
        projmatrix=viewpoint_camera.full_proj_transform,
        sh_degree=pc.active_sh_degree,
        sh_degree_t=pc.active_sh_degree_t,
        campos=viewpoint_camera.camera_center,
        timestamp=viewpoint_camera.timestamp,
        time_duration=pc.time_duration[1]-pc.time_duration[0],
        rot_4d=pc.rot_4d,
        gaussian_dim=pc.gaussian_dim,
        force_sh_3d=pc.force_sh_3d,
        prefiltered=False,
        opa_threshold=pipe.opa_threshold,
        debug=pipe.debug
    )

    rasterizer = GaussianRasterizer(raster_settings=raster_settings)

    means3D = pc.get_xyz
    means2D = screenspace_points
    opacity = pc.get_opacity

    # If precomputed 3d covariance is provided, use it. If not, then it will be computed from
    # scaling / rotation by the rasterizer.
    scales = None
    scales_t = None
    rotations = None
    rotations_r = None
    ts = None
    cov3D_precomp = None
    if pipe.compute_cov3D_python:
        if pc.rot_4d:
            cov3D_precomp, delta_mean = pc.get_current_covariance_and_mean_offset(scaling_modifier, viewpoint_camera.timestamp)
            means3D = means3D + delta_mean
        else:
            cov3D_precomp = pc.get_covariance(scaling_modifier)
        if pc.gaussian_dim == 4:
            marginal_t = pc.get_marginal_t(viewpoint_camera.timestamp)
            # marginal_t = torch.clamp_max(marginal_t, 1.0) # NOTE: 这里乘完会大于1，绝对不行——marginal_t应该用个概率而非概率密度 暂时可以clamp一下，后期用积分 —— 2d 也用的clamp
            opacity = opacity * marginal_t
    else:
        scales = pc.get_scaling
        rotations = pc.get_rotation
        if pc.gaussian_dim == 4:
            scales_t = pc.get_scaling_t
            ts = pc.get_t
            if pc.rot_4d:
                rotations_r = pc.get_rotation_r

    # If precomputed colors are provided, use them. Otherwise, if it is desired to precompute colors
    # from SHs in Python, do it. If not, then SH -> RGB conversion will be done by rasterizer.
    shs = None
    colors_precomp = None
    if override_color is None:
        if pipe.convert_SHs_python:
            shs_view = pc.get_features.transpose(1, 2).view(-1, 3, pc.get_max_sh_channels)
            if pipe.compute_cov3D_python:
                dir_pp = (means3D - viewpoint_camera.camera_center.repeat(pc.get_features.shape[0], 1)).detach()
            else:
                _, delta_mean = pc.get_current_covariance_and_mean_offset(scaling_modifier, viewpoint_camera.timestamp)
                dir_pp = ((means3D + delta_mean) - viewpoint_camera.camera_center.repeat(pc.get_features.shape[0], 1)).detach()
            dir_pp_normalized = dir_pp/dir_pp.norm(dim=1, keepdim=True)
            if pc.gaussian_dim == 3 or pc.force_sh_3d:
                sh2rgb = eval_sh(pc.active_sh_degree, shs_view, dir_pp_normalized)
            elif pc.gaussian_dim == 4:
                dir_t = (pc.get_t - viewpoint_camera.timestamp).detach()
                sh2rgb = eval_shfs_4d(pc.active_sh_degree, pc.active_sh_degree_t, shs_view, dir_pp_normalized, dir_t, pc.time_duration[1] - pc.time_duration[0])
            colors_precomp = torch.clamp_min(sh2rgb + 0.5, 0.0)
        else:
            shs = pc.get_features
            if pc.gaussian_dim == 4 and ts is None:
                ts = pc.get_t
    else:
        colors_precomp = override_color
    
    flow_2d = torch.zeros_like(pc.get_xyz[:,:2])
    
    # Prefilter
    if pipe.compute_cov3D_python and pc.gaussian_dim == 4:
        mask = marginal_t[:,0] > 0.05
        if means2D is not None:
            means2D = means2D[mask]
        if means3D is not None:
            means3D = means3D[mask]
        if ts is not None:
            ts = ts[mask]
        if shs is not None:
            shs = shs[mask]
        if colors_precomp is not None:
            colors_precomp = colors_precomp[mask]
        if opacity is not None:
            opacity = opacity[mask]
        if scales is not None:
            scales = scales[mask]
        if scales_t is not None:
            scales_t = scales_t[mask]
        if rotations is not None:
            rotations = rotations[mask]
        if rotations_r is not None:
            rotations_r = rotations_r[mask]
        if cov3D_precomp is not None:
            cov3D_precomp = cov3D_precomp[mask]
        if flow_2d is not None:
            flow_2d = flow_2d[mask]
    
    # Rasterize visible Gaussians to image, obtain their radii (on screen). 

    means3D_static = pc.get_static_xyz
    screenspace_points_static = torch.zeros_like(means3D_static, dtype=means3D_static.dtype, requires_grad=True, device="cuda") + 0
    try:
        screenspace_points_static.retain_grad()
    except:
        pass
    means2D_static = screenspace_points_static
    opacity_static = pc.get_static_opacity
    sh_static = pc.get_static_features
    scales_static = pc.get_static_scaling
    rotations_static = pc.get_static_rotation 

    rendered_image, radii, depth, alpha, flow, covs_com, radii_static, color_4d, color_3d, invdepth = rasterizer(
        means3D = means3D,
        means2D = means2D,
        shs = shs,
        colors_precomp = colors_precomp,
        flow_2d = flow_2d,
        opacities = opacity,
        ts = ts,
        scales = scales,
        scales_t = scales_t,
        rotations = rotations,
        rotations_r = rotations_r,
        cov3D_precomp = cov3D_precomp,
        means3D_static = means3D_static,
        means2D_static = means2D_static,
        shs_static = sh_static,
        opacities_static = opacity_static,
        scales_static = scales_static,
        rotations_static = rotations_static)
    

        # rendered_image, radii, depth, alpha, flow, covs_com = rasterizer(
        #     means3D = means3D,
        #     means2D = means2D,
        #     shs = shs,
        #     colors_precomp = colors_precomp,
        #     flow_2d = flow_2d,
        #     opacities = opacity,
        #     ts = ts,
        #     scales = scales,
        #     scales_t = scales_t,
        #     rotations = rotations,
        #     rotations_r = rotations_r,
        #     cov3D_precomp = cov3D_precomp)
    
    if pipe.env_map_res:
        assert pc.env_map is not None
        R = 60
        rays_o, rays_d = viewpoint_camera.get_rays()
        delta = ((rays_o*rays_d).sum(-1))**2 - (rays_d**2).sum(-1)*((rays_o**2).sum(-1)-R**2)
        assert (delta > 0).all()
        t_inter = -(rays_o*rays_d).sum(-1)+torch.sqrt(delta)/(rays_d**2).sum(-1)
        xyz_inter = rays_o + rays_d * t_inter.unsqueeze(-1)
        tu = torch.atan2(xyz_inter[...,1:2], xyz_inter[...,0:1]) / (2 * torch.pi) + 0.5 # theta
        tv = torch.acos(xyz_inter[...,2:3] / R) / torch.pi
        texcoord = torch.cat([tu, tv], dim=-1) * 2 - 1
        bg_color_from_envmap = F.grid_sample(pc.env_map[None], texcoord[None])[0] # 3,H,W
        # mask2 = (0 < xyz_inter[...,0]) & (xyz_inter[...,1] > 0) # & (xyz_inter[...,2] > -19)
        rendered_image = rendered_image + (1 - alpha) * bg_color_from_envmap # * mask2[None]
    
    if pipe.compute_cov3D_python and pc.gaussian_dim == 4:
        radii_all = radii.new_zeros(mask.shape)
        radii_all[mask] = radii
    else:
        radii_all = radii

    # Those Gaussians that were frustum culled or had a radius of 0 were not visible.
    # They will be excluded from value updates used in the splitting criteria.
    return {"render": rendered_image,
            "viewspace_points": screenspace_points,
            "visibility_filter" : radii_all > 0,
            "radii": radii_all,
            "depth": depth,
            "alpha": alpha,
            "flow": flow,
            "radii_static": radii_static,
            "visibility_filter_static": radii_static > 0,
            "render_4d": color_4d,
            "render_3d": color_3d,
            "viewspace_points_static": screenspace_points_static}


- arguments.py

- scene/__init__.py

- render.py

- gaussian_renderer/diff_gaussian_rasterization.py

- diff-gaussian-rasterization/cuda_rasterizer/forward.h

- diff-gaussian-rasterization/cuda_rasterizer/forward.cu

# restore 함수

In [ ]:
def restore(self, model_args, training_args):
        if self.gaussian_dim == 3:
            (self.active_sh_degree,
            self._xyz,
            self._features_dc,
            self._features_rest,
            self._scaling,
            self._rotation,
            self._opacity,
            self.max_radii2D,
            xyz_gradient_accum,
            denom,
            opt_dict,
            self.spatial_lr_scale) = model_args
        elif self.gaussian_dim == 4:
            (self.active_sh_degree,
            self._xyz,
            self._features_dc,
            self._features_rest,
            self._scaling,
            self._rotation,
            self._opacity,
            self.max_radii2D,
            xyz_gradient_accum,
            t_gradient_accum,
            denom,
            opt_dict,
            self.spatial_lr_scale,
            self._t,
            self._scaling_t,
            self._rotation_r,
            self.rot_4d,
            self.env_map,
            self.active_sh_degree_t,
            self.static_xyz,
            self.static_features_dc,
            self.static_features_rest,
            self.static_scaling,
            self.static_rotation,
            self.static_opacity,
            self.static_max_radii2D,
            self.static_denom,
            self.static_xyz_gradient_accum) = model_args
        if training_args is not None:
            self.training_setup(training_args)
            self.xyz_gradient_accum = xyz_gradient_accum
            self.t_gradient_accum = t_gradient_accum
            self.denom = denom
            self.optimizer.load_state_dict(opt_dict)